# 本地日线档案：Python、SQL 和 CSV
只读指定快照，不取数，不生成信号或回测。请使用 quant/.venv 的 Python 内核。字段、布局和权限说明见 [数据使用手册](../DATA.md)。


In [ ]:
import json
import os
from datetime import date
from pathlib import Path

from hetu_data import DataStore
from hetu_data.archive import ARCHIVE_COLUMNS, open_archive_view, query_archive
from hetu_data import SnapshotRef

# 可直接修改下面的值；环境变量方便脚本重放同一次查询。
root = Path(os.environ.get("HETU_QUANT_ROOT", ".hetu/quant")).resolve()
snapshot_id = os.environ.get("HETU_ARCHIVE_SNAPSHOT", "请填写已发布快照ID")
params = json.loads(os.environ.get(
    "HETU_ARCHIVE_QUERY", '{"codes":["600519.SH"],"start":"2026-01-01","end":"2026-10-13"}'
))
store = DataStore(root)
ref = SnapshotRef(snapshot_id=snapshot_id, root=root)
selection = dict(
    codes=tuple(params["codes"]), start=date.fromisoformat(params["start"]), end=date.fromisoformat(params["end"])
)


In [ ]:
result = query_archive(store, ref, **selection)
bars, gaps = result["bars"], result["gaps"]
bars


In [ ]:
# 即使日线为空，也保留未上市、退市、停牌、日历未知等解释。
gaps


SQL 视图映射同一清单中的 Parquet；没有复制整表到数据库。视图不可直接修改源数据，但连接并非任意 SQL 的安全沙箱。SQL 示例全视图读取权不足时会明确拒绝；Python 可查询授权子范围。


In [ ]:
with open_archive_view(store, ref) as con:
    sql_bars = con.execute(
        "SELECT " + ",".join(ARCHIVE_COLUMNS) + " FROM daily_bars "
        "WHERE code IN (SELECT unnest(?)) AND date BETWEEN ? AND ? ORDER BY code,date,product_id,source_ref",
        [list(selection["codes"]), params["start"], params["end"]],
    ).df()
import pandas as pd

pd.testing.assert_frame_equal(bars, sql_bars)
sql_bars


In [ ]:
# 与网页下载使用相同字段、顺序、口径；需要时自行保存到新文件。
csv_bytes = bars.to_csv(index=False).encode("utf-8-sig")
# Path("my-query.csv").write_bytes(csv_bytes)
